[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonardorochaperazzini/mvp-puc-rio-ml-analytics/blob/main/mvp_chicago_crimes_ml.ipynb)

# MVP — Machine Learning & Analytics

**PUC-Rio | Pós-graduação em Ciência de Dados e Analytics**  
**Disciplina:** Sprint: Machine Learning & Analytics  
**Nome:** Leonardo Rocha Perazzini  
**Dataset:** [Chicago Crimes 2012–2017 — Kaggle](https://www.kaggle.com/datasets/currie32/crimes-in-chicago)  
**Tipo de problema:** Classificação binária

---

# 1. Definição do Problema

## 1.1 Contexto

Chicago é uma das maiores metrópoles dos EUA e o Chicago Police Department registra todas as ocorrências criminais no sistema CLEAR desde 2001. Este MVP dá continuidade ao trabalho de Análise de Dados e Boas Práticas sobre o mesmo dataset, avançando para a etapa de modelagem preditiva.

## 1.2 Objetivo

> Construir e avaliar modelos de Machine Learning para prever se um crime registrado em Chicago resultará em prisão (`Arrest = True`), comparando baseline com modelos candidatos e discutindo limitações e oportunidades de melhoria.

## 1.3 Tipo de problema

**Classificação binária** — a variável-alvo `Arrest` assume dois valores discretos: `True` (houve prisão) ou `False` (não houve). A saída é uma categoria discreta, não um valor contínuo — classificação supervisionada é a abordagem natural.

## 1.4 Motivação para Machine Learning

As hipóteses levantadas no MVP anterior mostram que a probabilidade de prisão depende de múltiplos fatores correlacionados (tipo de crime, local, horário, natureza doméstica, distrito). Regras manuais não capturam essas interações — modelos de ML são mais adequados para encontrar padrões em ~291k registros com dezenas de variáveis.

## 1.5 Hipóteses e critério de sucesso

A análise exploratória do MVP anterior revelou padrões que motivam as features deste MVP:

| Hipótese | Achado | Impacto na modelagem |
|---|---|---|
| H1: `Primary Type` determina prisão | Confirmada — NARCOTICS 99% vs. ASSAULT 23% | Feature mais importante esperada |
| H2: `Domestic` aumenta chance de prisão | Refutada — domésticos têm *menor* taxa (20% vs. 27%) | Feature incluída, efeito inverso |
| H3: `Hour` influencia a taxa | Parcial — noite 31% vs. madrugada 21% | Features temporais relevantes |
| H4: `Year` importa | Confirmada — colapso pós-2015 (caso Laquan McDonald) | Incluída como tendência temporal |

**Critério de sucesso:** F1-weighted no conjunto de teste superior ao baseline em pelo menos 15 pontos percentuais.

---

# 2. Ambiente e Reprodutibilidade

In [ ]:
# Garantia de versão do xgboost no Colab
!pip install xgboost -q

In [ ]:
import os
import sys
import time
import warnings
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay
)
from scipy.stats import randint, uniform
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Sklearn:", __import__("sklearn").__version__)
print("XGBoost:", __import__("xgboost").__version__)
print("Seed:", SEED)

In [ ]:
def evaluate_model(y_true, y_pred, proba=None, train_time=0.0):
    result = {
        "accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "f1_weighted": round(f1_score(y_true, y_pred, average="weighted"), 4),
    }
    try:
        result["roc_auc"] = round(roc_auc_score(y_true, proba[:, 1]), 4) if proba is not None else float("nan")
    except Exception:
        result["roc_auc"] = float("nan")
    result["train_time_s"] = round(train_time, 1)
    return result

def show_results(results_dict):
    return pd.DataFrame(results_dict).T.sort_values("f1_weighted", ascending=False)

---

# 3. Apresentação e Carga dos Dados

## 3.1 Fonte

- **Dataset:** Chicago Crimes 2012–2017, publicado no Kaggle por David Currie
- **Amostra:** 20% estratificada (~291k registros) gerada com `random_state=42` no MVP anterior
- **Carga:** via URL pública do GitHub (sem login, token ou upload manual)
- **Variável-alvo:** `Arrest` — se o crime resultou em prisão (True/False)
- **Atributos originais:** 22 colunas (identificadores, temporal, geográfico, tipo, local, flags)

## 3.2 Limitações do dataset

- Cobre apenas 2012–2017; padrões podem ter mudado após esse período
- Apenas crimes *registrados* — crimes não reportados não aparecem
- A taxa de prisão reflete também eficiência policial, não apenas o comportamento criminal
- Informações geográficas finas (Latitude, Longitude) serão descartadas por dificuldade de imputação

In [ ]:
GITHUB_CSV = (
    "https://raw.githubusercontent.com/leonardorochaperazzini/"
    "mvp-puc-rio-ml-analytics/main/data/Chicago_Crimes_2012_to_2017_sample.csv"
)
LOCAL_CSV = "data/Chicago_Crimes_2012_to_2017_sample.csv"

try:
    df = pd.read_csv(GITHUB_CSV)
    print(f"Dataset carregado do GitHub: {df.shape}")
except Exception:
    df = pd.read_csv(LOCAL_CSV)
    print(f"Dataset carregado localmente: {df.shape}")

df.head(3)

In [ ]:
print("Formato:", df.shape)
print("\nTipos de dados:")
display(df.dtypes.to_frame("tipo"))
print("\nValores nulos:")
nulos = df.isnull().sum()
display(nulos[nulos > 0].to_frame("nulos"))

In [ ]:
print("Estatísticas descritivas (numéricas):")
display(df.describe())

---

# 4. Análise Exploratória

A EDA completa foi realizada no MVP anterior (Sprint de Análise de Dados e Boas Práticas). Esta seção apresenta apenas os achados diretamente relevantes para as decisões de modelagem.

In [ ]:
# Distribuição do target
target_counts = df["Arrest"].value_counts()
target_pct = df["Arrest"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(
    x=target_counts.index.astype(str), y=target_counts.values,
    palette=["#e74c3c", "#2ecc71"], ax=axes[0]
)
axes[0].set_title("Distribuição de Arrest (contagem)")
axes[0].set_xlabel("Arrest")
axes[0].set_ylabel("Ocorrências")
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 500, f"{v:,}", ha="center", fontsize=10)

axes[1].pie(
    target_pct.values,
    labels=[f"{l} ({v:.1f}%)" for l, v in zip(target_counts.index.astype(str), target_pct.values)],
    colors=["#e74c3c", "#2ecc71"], startangle=90, autopct=None
)
axes[1].set_title("Proporção Arrest")
plt.tight_layout()
plt.show()

print(f"Desbalanceamento: {target_counts[False]:,} False ({target_pct[False]:.1f}%) "
      f"vs {target_counts[True]:,} True ({target_pct[True]:.1f}%)")
print(f"scale_pos_weight para XGBoost: {target_counts[False]/target_counts[True]:.2f}")

In [ ]:
# Taxa de prisão por tipo de crime — motivação para Primary Type como feature principal
arrest_by_type = (
    df.groupby("Primary Type")["Arrest"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)
arrest_by_type.columns = ["Primary Type", "Arrest Rate (%)"]

plt.figure(figsize=(10, 6))
sns.barplot(data=arrest_by_type, x="Arrest Rate (%)", y="Primary Type", palette="Reds_r")
plt.title("Taxa de Prisão por Tipo de Crime (Top 15) — confirma H1")
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição temporal — motivação para Hour e Year como features
df_eda = df.copy()
df_eda["Date_dt"] = pd.to_datetime(df_eda["Date"], format="%m/%d/%Y %I:%M:%S %p")
df_eda["Hour"] = df_eda["Date_dt"].dt.hour
df_eda["Year_eda"] = df_eda["Date_dt"].dt.year

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hourly = df_eda.groupby("Hour")["Arrest"].mean().mul(100).reset_index()
hourly.columns = ["Hour", "Arrest Rate (%)"]
sns.lineplot(data=hourly, x="Hour", y="Arrest Rate (%)", marker="o", ax=axes[0])
axes[0].set_title("Taxa de Prisão por Hora do Dia — confirma H3")

yearly = df_eda.groupby("Year_eda")["Arrest"].mean().mul(100).reset_index()
yearly.columns = ["Year", "Arrest Rate (%)"]
sns.lineplot(data=yearly, x="Year", y="Arrest Rate (%)", marker="o", ax=axes[1])
axes[1].set_title("Taxa de Prisão por Ano — colapso pós-2015 (H4)")

plt.tight_layout()
plt.show()

del df_eda

## 4.1 Síntese e impacto nas decisões de modelagem

| Achado | Impacto |
|---|---|
| `Primary Type` determina fortemente a prisão (H1) | Feature nominal com ~35 valores → OneHotEncoder |
| `Domestic` tem relação inversa com prisão (H2) | Incluída como feature binária |
| `Hour` influencia a taxa de prisão (H3) | Extraída de `Date` como feature numérica |
| `Year` marca colapso pós-2015 (H4) | Incluída como feature numérica |
| Target desbalanceado (~74% False / ~26% True) | `class_weight='balanced'` (sklearn) e `scale_pos_weight` (XGBoost) |

---

# 5. Preparação dos Dados

## 5.1 Feature engineering

In [ ]:
df_model = df.copy()

# Extrair features temporais de Date (string → datetime → componentes)
df_model["Date_dt"] = pd.to_datetime(df_model["Date"], format="%m/%d/%Y %I:%M:%S %p")
df_model["Hour"]      = df_model["Date_dt"].dt.hour
df_model["DayOfWeek"] = df_model["Date_dt"].dt.dayofweek  # 0=Segunda, 6=Domingo
df_model["Month"]     = df_model["Date_dt"].dt.month

# Converter target bool → int (0/1 para compatibilidade com XGBoost)
df_model["Arrest"] = df_model["Arrest"].astype(int)

# Colunas descartadas e justificativa
COLS_DROP = [
    "ID", "Case Number",           # identificadores sem valor preditivo
    "Date", "Date_dt",             # substituídas pelas features temporais extraídas
    "Block", "IUCR", "Description", "FBI Code",  # redundantes ou muito granulares
    "X Coordinate", "Y Coordinate", "Latitude", "Longitude", "Location",  # geográficas com muitos nulos
    "Updated On",                  # data de atualização do registro, não do crime
    "Beat",                        # subunidade de District — redundante
    "Ward",                        # divisão política, correlacionada com District
]
df_model = df_model.drop(columns=COLS_DROP, errors="ignore")

print("Colunas restantes:", df_model.columns.tolist())
print("Shape:", df_model.shape)

## 5.2 Features selecionadas

| Feature | Tipo no pipeline | Justificativa |
|---|---|---|
| `Primary Type` | Categórica | Maior preditor de prisão (H1) — alta cardinalidade tratada com OneHot |
| `Location Description` | Categórica | Local influencia abordagem policial |
| `District` | Categórica (nominal) | Código nominal — OneHot, não escalar |
| `Community Area` | Categórica (nominal) | Código nominal — OneHot, não escalar |
| `Domestic` | Numérica (0/1) | Relação inversa com prisão (H2) |
| `Year` | Numérica | Tendência temporal (H4) |
| `Hour` | Numérica | Padrão intradiário (H3) |
| `DayOfWeek` | Numérica | Variação por dia da semana |
| `Month` | Numérica | Sazonalidade |

> **Nota:** `District` e `Community Area` são códigos nominais (ex: Distrito 7 não é "maior que" Distrito 3). Por isso usamos `OneHotEncoder`, não `StandardScaler`.

## 5.3 Divisão treino / teste

In [ ]:
TARGET = "Arrest"
features = [c for c in df_model.columns if c != TARGET]

X = df_model[features].copy()
y = df_model[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Proporção do target no treino: {y_train.mean():.3f}")
print(f"Proporção do target no teste:  {y_test.mean():.3f}")
print("\nEstratificação manteve a proporção — sem data leakage")

---

# 6. Pipeline de Pré-processamento

In [ ]:
NUM_COLS = ["Domestic", "Year", "Hour", "DayOfWeek", "Month"]
CAT_COLS = ["Primary Type", "Location Description", "District", "Community Area"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, NUM_COLS),
    ("cat", categorical_pipe, CAT_COLS),
], remainder="drop")

print("Colunas numéricas:", NUM_COLS)
print("Colunas categóricas:", CAT_COLS)

## 6.1 Decisões de pré-processamento

| Decisão | Justificativa |
|---|---|
| Imputação pela mediana (numéricas) | Robusta a outliers; nulos em Year/Hour são raros |
| Imputação pela moda (categóricas) | Preserva a categoria mais frequente para raros nulos em Location Description |
| `StandardScaler` nas numéricas | Necessário para `LogisticRegression` convergir; neutro para árvores |
| `OneHotEncoder` nas nominais | District e Community Area são códigos sem ordem — tratar como numérico seria erro de modelagem |
| `fit` apenas no treino | Evita vazamento: parâmetros do scaler/encoder não "veem" o conjunto de teste |
| `handle_unknown='ignore'` | Garante robustez a categorias raras que aparecem apenas no teste |

> O pipeline resulta em ~260 colunas binárias após OneHot — normal e eficiente para modelos baseados em árvores.

---

# 7. Modelagem

## 7.1 Tratamento do desbalanceamento

In [ ]:
SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight (XGBoost): {SCALE_POS_WEIGHT:.2f}")
print(f"Equivalente a class_weight='balanced' nos modelos sklearn")

## 7.2 Definição dos modelos

| Modelo | Papel | Justificativa |
|---|---|---|
| `DummyClassifier` | Baseline | Sempre prevê a classe majoritária. Referência mínima — qualquer modelo útil deve superá-lo. |
| `LogisticRegression` | Candidato linear | Modelo interpretável, rápido. Bom ponto de partida para classificação binária. Requer escala. |
| `DecisionTree` | Candidato interpretável | Árvore simples — expõe o que foi aprendido. Ponte entre baseline e ensembles. |
| `RandomForest` | Candidato ensemble-bagging | Reduz variância via averaging de múltiplas árvores. Robusto e consistente. |
| `XGBoost` | Candidato ensemble-boosting | Treina sequencialmente corrigindo erros. Estado da arte em dados tabulares. |

In [ ]:
baseline = Pipeline([
    ("preprocess", preprocess),
    ("model", DummyClassifier(strategy="most_frequent", random_state=SEED))
])

candidates = {
    "LogisticRegression": Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(
            max_iter=1000, random_state=SEED, class_weight="balanced"
        ))
    ]),
    "DecisionTree": Pipeline([
        ("preprocess", preprocess),
        ("model", DecisionTreeClassifier(
            max_depth=10, random_state=SEED, class_weight="balanced"
        ))
    ]),
    "RandomForest": Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=100, random_state=SEED, class_weight="balanced", n_jobs=-1
        ))
    ]),
    "XGBoost": Pipeline([
        ("preprocess", preprocess),
        ("model", XGBClassifier(
            n_estimators=100,
            scale_pos_weight=SCALE_POS_WEIGHT,
            random_state=SEED,
            eval_metric="logloss",
            verbosity=0,
            n_jobs=-1
        ))
    ]),
}

print("Modelos definidos:", ["baseline"] + list(candidates.keys()))

---

# 8. Treinamento e Avaliação Inicial

In [ ]:
results = {}
trained_models = {}

# Baseline
t0 = time.time()
baseline.fit(X_train, y_train)
t_baseline = time.time() - t0
y_pred_bl = baseline.predict(X_test)
results["Baseline"] = evaluate_model(y_test, y_pred_bl, train_time=t_baseline)
trained_models["Baseline"] = baseline
print(f"Baseline concluído | F1={results['Baseline']['f1_weighted']:.4f}")

# Candidatos
for name, model in candidates.items():
    print(f"Treinando {name}...", end=" ")
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0
    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    results[name] = evaluate_model(y_test, y_pred, proba, train_time=elapsed)
    trained_models[name] = model
    print(f"concluído em {elapsed:.1f}s | F1={results[name]['f1_weighted']:.4f}")

print("\nResultados iniciais:")
display(show_results(results))

## 8.1 Análise dos resultados iniciais

Os resultados acima mostram a primeira comparação entre os modelos. Pontos a observar:

- **Baseline (Dummy):** prevê sempre `False` (classe majoritária) — F1-weighted alto pela precisão na classe dominante, mas ROC-AUC = 0.5 (aleatório). É o piso de comparação.
- **LogisticRegression:** modelo linear; esperado ser mais fraco que ensembles em relações não-lineares.
- **DecisionTree:** uma única árvore pode ser propensa a overfitting; espera-se menor generalização que RF.
- **RandomForest e XGBoost:** ensembles — esperamos os melhores resultados antes da otimização.

> A otimização de hiperparâmetros na seção seguinte busca extrair o máximo do XGBoost.

---

# 9. Otimização de Hiperparâmetros

## 9.1 Estratégia

Aplicamos `RandomizedSearchCV` no XGBoost, que demonstrou o melhor potencial nos resultados iniciais.

- **Método:** RandomizedSearchCV (mais eficiente que GridSearch em espaços contínuos)
- **Validação:** StratifiedKFold com 3 folds (preserva proporção das classes)
- **n_iter=10, cv=3** → 30 fits totais (~5–8 min no Colab)
- **Métrica de seleção:** F1-weighted
- **Prevenção de data leakage:** busca feita *apenas* no treino; teste intocado

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

model_to_tune = Pipeline([
    ("preprocess", preprocess),
    ("model", XGBClassifier(
        scale_pos_weight=SCALE_POS_WEIGHT,
        random_state=SEED,
        eval_metric="logloss",
        verbosity=0,
        n_jobs=-1
    ))
])

param_dist = {
    "model__n_estimators":  randint(100, 301),
    "model__max_depth":     randint(3, 9),
    "model__learning_rate": uniform(0.01, 0.29),
    "model__subsample":     uniform(0.6, 0.4),
}

search = RandomizedSearchCV(
    model_to_tune,
    param_distributions=param_dist,
    n_iter=10,
    cv=cv,
    scoring="f1_weighted",
    random_state=SEED,
    n_jobs=1,
    verbose=1,
    refit=True
)

print("Iniciando busca... (estimativa: 5–8 min)")
t0 = time.time()
search.fit(X_train, y_train)
elapsed_search = time.time() - t0

print(f"\nBusca concluída em {elapsed_search/60:.1f} min")
print(f"Melhor F1-weighted (validação cruzada): {search.best_score_:.4f}")
print(f"Melhores hiperparâmetros:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

In [ ]:
# Avaliação do XGBoost otimizado no conjunto de teste
y_pred_opt = search.best_estimator_.predict(X_test)
proba_opt  = search.best_estimator_.predict_proba(X_test)

results["XGBoost_otimizado"] = evaluate_model(
    y_test, y_pred_opt, proba_opt, train_time=elapsed_search
)
trained_models["XGBoost_otimizado"] = search.best_estimator_

print("Comparação XGBoost base vs. otimizado:")
display(pd.DataFrame({
    k: results[k] for k in ["XGBoost", "XGBoost_otimizado"]
}).T)

## 9.2 Discussão da otimização

A busca aleatória explorou o espaço de hiperparâmetros de forma eficiente:

- **`n_estimators`:** mais árvores geralmente reduzem variância até um ponto de saturação
- **`max_depth`:** controla a complexidade de cada árvore — valores menores reduzem overfitting
- **`learning_rate`:** taxa de aprendizado; valores menores requerem mais estimadores para convergir
- **`subsample`:** fração de amostras por árvore — adiciona ruído controlado (regularização implícita)

> A otimização foi feita com validação cruzada *apenas* no treino — o conjunto de teste permaneceu intocado durante toda a busca, garantindo uma avaliação honesta.

---

# 10. Avaliação Final

In [ ]:
# Identificar o melhor modelo
results_df = show_results(results)
best_model_name = results_df.index[0]
final_model = trained_models[best_model_name]

print(f"Modelo final: {best_model_name}")
print(f"F1-weighted no teste: {results_df.loc[best_model_name, 'f1_weighted']:.4f}")
print(f"ROC-AUC no teste:     {results_df.loc[best_model_name, 'roc_auc']:.4f}")
print()

y_pred_final = final_model.predict(X_test)
print(classification_report(y_test, y_pred_final, target_names=["Não Preso", "Preso"]))

In [ ]:
# Matriz de confusão
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    final_model, X_test, y_test,
    display_labels=["Não Preso", "Preso"],
    cmap="Blues", ax=ax
)
ax.set_title(f"Matriz de Confusão — {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — modelos baseados em árvores
model_step = final_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    try:
        feature_names = final_model.named_steps["preprocess"].get_feature_names_out()
    except Exception:
        feature_names = [f"feature_{i}" for i in range(len(model_step.feature_importances_))]

    importance_df = pd.DataFrame({
        "feature":    feature_names,
        "importance": model_step.feature_importances_
    }).sort_values("importance", ascending=False).head(20)

    plt.figure(figsize=(10, 7))
    sns.barplot(data=importance_df, x="importance", y="feature", palette="Blues_r")
    plt.title(f"Top 20 Features Mais Importantes — {best_model_name}")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance não disponível para este modelo.")

In [ ]:
# Análise de overfitting: comparação treino vs. teste
y_pred_train = final_model.predict(X_train)
f1_train = f1_score(y_train, y_pred_train, average="weighted")
f1_test  = results_df.loc[best_model_name, "f1_weighted"]

print(f"F1-weighted no treino: {f1_train:.4f}")
print(f"F1-weighted no teste:  {f1_test:.4f}")
print(f"Gap treino-teste:      {f1_train - f1_test:.4f}")

if f1_train - f1_test > 0.05:
    print("Gap > 5pp — possível overfitting. Considere regularização adicional.")
else:
    print("Gap dentro do esperado — sem sinal forte de overfitting.")

## 10.1 Análise de erros e limitações

A matriz de confusão mostra dois tipos de erro:

- **Falsos negativos (FN):** crimes que resultaram em prisão mas o modelo previu que não — o modelo subestima prisões
- **Falsos positivos (FP):** crimes que *não* resultaram em prisão mas o modelo previu que sim — o modelo sobrestima prisões

O `class_weight='balanced'` / `scale_pos_weight` reduz FN ao custo de aumentar FP — trade-off adequado para o problema (identificar casos de prisão é mais importante que minimizar alarmes falsos).

**Limitações do modelo:**
- Não deve ser usado para decisões reais de policiamento — é um exercício acadêmico
- Viés histórico: o modelo aprende padrões de discriminação sistêmica presentes nos dados
- Dados de 2012–2017 podem não refletir dinâmicas atuais
- Crimes não reportados (cifra negra) não estão representados

---

# 11. Comparação Final

In [ ]:
print("Comparação final — todos os modelos (ordenado por F1-weighted):")
display(show_results(results))

In [ ]:
# Visualização comparativa
results_viz = show_results(results)[["f1_weighted", "roc_auc"]].reset_index()
results_viz.columns = ["Modelo", "F1-weighted", "ROC-AUC"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=results_viz, x="F1-weighted", y="Modelo", palette="Blues_r", ax=axes[0])
axes[0].set_title("F1-weighted por Modelo")
axes[0].set_xlim(0, 1)

sns.barplot(data=results_viz.dropna(subset=["ROC-AUC"]), x="ROC-AUC", y="Modelo", palette="Greens_r", ax=axes[1])
axes[1].set_title("ROC-AUC por Modelo")
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

---

# 12. Boas Práticas e Rastreabilidade

In [ ]:
print("=" * 50)
print("REGISTRO DE DECISÕES E REPRODUTIBILIDADE")
print("=" * 50)
print(f"Seed global:              {SEED}")
print(f"Split:                    80% treino / 20% teste, estratificado")
print(f"Desbalanceamento:         class_weight='balanced' (sklearn)")
print(f"                          scale_pos_weight={SCALE_POS_WEIGHT:.2f} (XGBoost)")
print(f"Pipeline fit:             apenas no treino (sem data leakage)")
print(f"Otimização:               RandomizedSearchCV, n_iter=10, cv=3")
print(f"                          limitado por custo computacional do Colab")
print(f"Sem arquivos intermediários: pipeline em memória")
print()
print("Bibliotecas utilizadas:")
import sklearn, xgboost, scipy, matplotlib
print(f"  pandas {pd.__version__} | numpy {np.__version__}")
print(f"  scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__}")
print(f"  scipy {scipy.__version__} | matplotlib {matplotlib.__version__}")

---

# 13. Conclusão

## Problema abordado

Este MVP abordou a **classificação binária** de crimes em Chicago para prever se uma ocorrência resultará em prisão, usando o dataset Chicago Crimes 2012–2017 (amostra estratificada de 20%, ~291k registros).

## Dataset e preparação

- Fonte: Chicago Crimes 2012–2017 via Kaggle / GitHub público
- 22 atributos originais → 9 features selecionadas após engenharia e remoção de redundâncias
- Feature engineering: extração de `Hour`, `DayOfWeek`, `Month` a partir da coluna `Date`
- Desbalanceamento (~74% False / ~26% True) tratado via `class_weight='balanced'` e `scale_pos_weight`
- Pipeline `ColumnTransformer` com `StandardScaler` para numéricas e `OneHotEncoder` para nominais (~260 colunas binárias)

## Modelos avaliados

Cinco modelos em progressão crescente de complexidade: `DummyClassifier` (baseline), `LogisticRegression`, `DecisionTree`, `RandomForest`, `XGBoost` (com versão padrão e otimizada via `RandomizedSearchCV`).

## Resultados

Os resultados acima mostram a comparação final. O critério de sucesso (F1-weighted 15pp acima do baseline) foi avaliado e o melhor modelo identificado. A feature importance confirmou a hipótese H1 — `Primary Type` domina as predições.

## Limitações

- Dataset cobre apenas 2012–2017; padrões podem ter mudado
- Amostra de 20% — resultados podem variar com o dataset completo
- Features geográficas descartadas — coordenadas poderiam melhorar o modelo com geocodificação adequada
- Otimização limitada a 10 iterações por restrição de tempo no Colab
- **Uso responsável:** este modelo aprende padrões históricos que podem conter viés sistêmico — não deve ser usado para decisões reais de policiamento

## Próximos passos

- Testar `LightGBM` com early stopping
- Explorar coordenadas geográficas com clusterização espacial (K-means) antes de incluir como feature
- Analisar calibração de probabilidades (Platt scaling / isotonic regression)
- Avaliar interpretabilidade com SHAP values para entender decisões individuais
- Ampliar busca de hiperparâmetros com Optuna em ambiente com mais recursos

---

# 14. Checklist do MVP

| Item | Status | Referência |
|---|---|---|
| **Definição do problema** | | |
| Objetivo claramente definido | ✅ | Seção 1.2 |
| Tipo de tarefa ML justificado (classificação binária) | ✅ | Seção 1.3 |
| Motivação para uso de ML | ✅ | Seção 1.4 |
| Premissas e hipóteses levantadas | ✅ | Seção 1.5 |
| **Dados** | | |
| Dataset descrito com fonte, atributos e restrições | ✅ | Seção 3 |
| Dataset carregado por URL pública (sem login/token) | ✅ | Seção 3 |
| Análise exploratória com distribuição da target | ✅ | Seção 4 |
| Desbalanceamento identificado e documentado | ✅ | Seção 4 |
| **Preparação** | | |
| Nulos tratados e justificados | ✅ | Seção 6 |
| Encoding adequado para cada tipo de variável | ✅ | Seção 6 |
| Normalização aplicada onde necessário | ✅ | Seção 6 |
| Feature engineering documentado | ✅ | Seção 5.1 |
| Prevenção de data leakage (fit apenas no treino) | ✅ | Seção 6 |
| **Modelagem** | | |
| Modelo baseline definido | ✅ | Seção 7 |
| Pelo menos 2 modelos além do baseline | ✅ | 4 candidatos |
| Comparação justa (mesmas métricas, mesmo split) | ✅ | Seção 8 |
| Justificativa para escolha dos modelos | ✅ | Seção 7.2 |
| **Otimização** | | |
| Hiperparâmetros ajustados em pelo menos 1 modelo | ✅ | Seção 9 |
| Estratégia de busca justificada | ✅ | Seção 9.1 |
| Otimização sem usar dados de teste | ✅ | Seção 9 |
| **Avaliação** | | |
| Métricas adequadas ao problema com justificativa | ✅ | F1-weighted + ROC-AUC |
| Análise de overfitting/underfitting | ✅ | Seção 10 |
| Análise de erros e limitações | ✅ | Seção 10.1 |
| Melhor modelo identificado e justificado | ✅ | Seção 11 |
| **Conclusão** | | |
| Conclusão conectada ao objetivo inicial | ✅ | Seção 13 |
| Próximos passos documentados | ✅ | Seção 13 |
| **Boas práticas** | | |
| Seed fixada para reprodutibilidade | ✅ | `SEED = 42` |
| Notebook roda do início ao fim sem erros | ✅ | |
| Código limpo e organizado | ✅ | |
| Bibliotecas informadas | ✅ | Seção 12 |
| Decisões documentadas em Markdown | ✅ | Todas as seções |